In [1]:
import pandas as pd

In [2]:
#read files
welfake_train= pd.read_csv("/content/drive/MyDrive/datasets/welfake_train.csv")
welfake_validation = pd.read_csv("/content/drive/MyDrive/datasets/welfake_validation.csv")
welfake_test = pd.read_csv("/content/drive/MyDrive/datasets/welfake_test.csv")

isot_train= pd.read_csv("/content/drive/MyDrive/datasets/isot_train.csv")
isot_validation = pd.read_csv("/content/drive/MyDrive/datasets/isot_validation.csv")
isot_test = pd.read_csv("/content/drive/MyDrive/datasets/isot_test.csv")

multifc_train = pd.read_csv("/content/drive/MyDrive/datasets/multifc_train.csv")
multifc_validation =pd.read_csv("/content/drive/MyDrive/datasets/multifc_validation.csv")
multifc_test =pd.read_csv("/content/drive/MyDrive/datasets/multifc_test.csv")

hard_examples= pd.read_csv("/content/drive/MyDrive/datasets/hard_examples_text_label_100.csv")


In [3]:
# Merge datasets
final_train = pd.concat(
    [welfake_train, isot_train, multifc_train, hard_examples],
    ignore_index=True
)

final_validation = pd.concat(
    [welfake_validation, isot_validation, multifc_validation],
    ignore_index=True
)

final_test = pd.concat(
    [welfake_test, isot_test, multifc_test],
    ignore_index=True
)

In [4]:
#remove duplicates
final_train = final_train.drop_duplicates(subset=["text"]).reset_index(drop=True)

final_validation = final_validation.drop_duplicates(subset=["text"]).reset_index(drop=True)

final_test = final_test.drop_duplicates(subset=["text"]).reset_index(drop=True)

In [5]:
#prevent data leakage for safty
def remove_leakage(train_df, val_df, test_df):

    train_texts = set(train_df["text"])

    val_df = val_df[~val_df["text"].isin(train_texts)].reset_index(drop=True)

    test_df = test_df[~test_df["text"].isin(train_texts)].reset_index(drop=True)

    val_texts = set(val_df["text"])

    test_df = test_df[~test_df["text"].isin(val_texts)].reset_index(drop=True)

    return train_df, val_df, test_df


final_train, final_validation, final_test = remove_leakage(
    final_train,
    final_validation,
    final_test
)

In [6]:
# Shuffle
final_train = final_train.sample(frac=1, random_state=42).reset_index(drop=True)
final_validation = final_validation.sample(frac=1, random_state=42).reset_index(drop=True)
final_test = final_test.sample(frac=1, random_state=42).reset_index(drop=True)

In [8]:
print(final_train.shape)
print(final_validation.shape)
print(final_test.shape)

(98820, 2)
(12108, 2)
(12580, 2)


In [9]:
#save files
final_train.to_csv("train_combined.csv", index=False)
final_validation.to_csv("validation_combined.csv", index=False)
final_test.to_csv("test_combined.csv", index=False)